In [1]:
import torch

In [2]:
torch.set_default_dtype(torch.float64)

In [3]:
m = 7
l = 9
vocab_size = 9
target = [0, 1, 2, 3, 4, 5, 8]
assert len(target) == m
target = torch.tensor(target)

In [4]:
transition_matrix = torch.zeros((l, l))

In [5]:
# the following are the coordinates and values, using 1-indexing
transitions = [
    (
        (1, 2), #row, col
        0.3 #prob
    ),
    (
        (1, 3), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        1.0 #prob
    ),
    (
        (3, 4), #row, col
        1.0 #prob
    ),
    (
        (4, 5), #row, col
        1.0 #prob
    ),
    (
        (5, 6), #row, col
        0.5 #prob
    ),
    (
        (5, 7), #row, col
        0.5 #prob
    ),
    (
        (6, 9), #row, col nice
        1.0 #prob
    ),
    (
        (7, 8), #row, col
        1.0 #prob
    ),
    (
        (8, 9), #row, col
        1.0 #prob
    ),
]

In [6]:
for (row, col), prob in transitions:
    transition_matrix[row-1, col-1] = prob

In [7]:
token_probs = torch.zeros((l, vocab_size))

In [8]:
# the following are the coordinates and values, using 1-indexing
# format is
# (row, col), value
# for example (1, 2), 0.8
# means state 1 emits token 2 with probability 0.8
emission_probs = [
    (
        (1, 1), #row, col
        0.9 #prob
    ),
    (
        (1, 5), #row, col
        0.1 #prob
    ),
    (
        (2, 2), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.3 #prob
    ),
    (
        (3, 2), #row, col
        0.2 #prob
    ),
    (
        (3, 3), #row, col
        0.8 #prob
    ),
    (
        (4, 3), #row, col
        0.1 #prob
    ),
    (
        (4, 4), #row, col
        0.9 #prob
    ),
    (
        (5, 4), #row, col
        0.1 #prob
    ),
    (
        (5, 5), #row, col
        0.9 #prob
    ),
    (
        (6, 6), #row, col
        0.6 #prob
    ),
    (
        (6, 7), #row, col
        0.3 #prob
    ),
    (
        (6, 8), #row, col
        0.1 #prob
    ),
    (
        (7, 5), #row, col
        0.1 #prob
    ),
    (
        (7, 6), #row, col
        0.1 #prob
    ),
    (
        (7, 7), #row, col
        0.7 #prob
    ),
    (
        (7, 2), #row, col
        0.1 #prob
    ),
    (
        (8, 6), #row, col
        0.2 #prob
    ),
    (
        (8, 8), #row, col
        0.6 #prob
    ),
    (
        (8, 9), #row, col
        0.2 #prob
    ),
    (
        (9, 8), #row, col
        0.3 #prob
    ),
    (
        (9, 9), #row, col
        0.7 #prob
    )
]

In [9]:
for (row, col), prob in emission_probs:
    token_probs[row-1, col-1] = prob

In [10]:
dp = torch.zeros((m, l))

In [11]:
dp[0, 0] = 1

In [12]:
i = 1

In [13]:
dp[i - 1, :] @ transition_matrix

tensor([0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

In [14]:
token_probs[:, target[i]]

tensor([0.0000, 0.7000, 0.2000, 0.0000, 0.0000, 0.0000, 0.1000, 0.0000, 0.0000])

In [15]:
token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

tensor([0.0000, 0.2100, 0.1400, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

In [16]:
dp[i, :] = token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

In [17]:
i = 2

In [18]:
dp[i, :] = token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

In [19]:
i = 3
dp[i, :] = token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

In [20]:
i = 4
dp[i, :] = token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

In [21]:
i = 5
dp[i, :] = token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

In [22]:
# print out dp current state rounded to 3 decimal places
print(dp.round(decimals=3))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.2100, 0.1400, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.1680, 0.0140, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.1510, 0.0010, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.1360, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0410, 0.0070, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])


In [23]:
i = 6

In [24]:
dp[i, :] = token_probs[:, target[i]] * (dp[i - 1, :] @ transition_matrix)

In [25]:
dp

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 2.1000e-01, 1.4000e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 1.6800e-01, 1.4000e-02, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.5120e-01, 1.4000e-03, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.3608e-01, 0.0000e+00,
         7.0000e-05, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 4.0824e-02,
         6.8040e-03, 1.4000e-05, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 1.3608e-03, 2.8587e-02]])

In [26]:
# doing it by hand by enumerating all possible paths
p1m = torch.tensor([0.9, 0.7, 0.8, 0.9, 0.9, 0.6, 0.7])
p1t = torch.tensor([0.3, 1, 1, 1, 0.5, 1])

p2m = torch.tensor([0.9, 0.7, 0.8, 0.9, 0.9, 0.1, 0.2])
p2t = torch.tensor([0.3, 1, 1, 1, 0.5, 1])

p3m = torch.tensor([0.9, 0.2, 0.1, 0.1, 0.1, 0.2, 0.7])
p3t = torch.tensor([0.7, 1, 1, 0.5, 1, 1])

In [32]:
p1 = p1m.prod() * p1t.prod()
p2 = p2m.prod() * p2t.prod()
p3 = p3m.prod() * p3t.prod()

In [33]:
acc = p1 + p2 + p3

In [34]:
acc

tensor(0.02695265999999999634)

In [35]:
# check difference between dp[m-1, l-1] and acc
dp[m-1, l-1] - acc

tensor(0.00163394000000000039)